# Recipe → Catalog matching: ghi chép pipeline (issue #8)

Notebook ghi lại **pipeline đã thống nhất**, phần **đã làm**, code giải thích, kết quả từng bước, **tự đánh giá** của AI và **hướng nâng cấp**.
Các cell code gọi lại đúng code trong `scripts/`, nên kết quả hiển thị là kết quả chạy thật trên dữ liệu hiện có.

> Mục tiêu cuối: người dùng gõ một đoạn text, hệ thống trả công thức giống nhất (top 1 nổi bật, 4 cái còn lại là lựa chọn khác),
> kèm một giỏ hàng nạp sẵn các nguyên liệu có trong catalog. Bấm một lần là bỏ cả giỏ vào.
> Giỏ phải **đủ nguyên liệu** và **đủ số lượng**.

| Bước | Nội dung | Trạng thái |
| :-- | :-- | :-: |
| 1 | Chuẩn hóa tên nguyên liệu trong công thức (`recipes_clean`) | ✅ làm xong |
| 2 | Chuẩn hóa catalog bằng cùng bộ luật (`catalog_clean`) | ✅ làm xong |
| 3 | bge-m3 gợi ý gộp tên đồng nghĩa và ghép nguyên liệu → catalog | ⏳ đã viết, đang chạy, **chưa duyệt tay** |
| 4 | Ghép chính thức, đo số công thức đủ nguyên liệu (option A) | ⬜ chưa làm |
| 5 | Bảng quy đổi đơn vị và quy tắc mua dư | ⬜ chưa làm |
| 6 | Embed công thức, truy vấn top-5 | ⬜ hoãn theo thống nhất |

## 0. Các quyết định đã thống nhất

- **Ghép offline.** Nguyên liệu → sản phẩm catalog được tính trước, kèm ID. Lúc chạy chỉ embed câu hỏi và lấy top-5 công thức. *(Khác proposal: proposal ghép online bằng embedding và ngưỡng δ. Cần ghi vào `docs/06`.)*
- **Catalog là nguồn sự thật duy nhất**, không đổi đơn vị catalog.
- **Số lượng: mua dư.** Quy đổi nguyên liệu sang gam/ml bằng bảng quy đổi (ví dụ 1 trái cà chua ≈ 40g), rồi **làm tròn lên** theo cỡ gói catalog.
- **Dạng nguyên liệu:** dạng *khó làm* (khô, đông lạnh, đóng hộp, ngâm chua) là mặt hàng riêng, giữ trong tên ("tôm khô", "cà chua hộp"). Dạng *dễ làm* (phi, xay, rang) gộp về tên gốc, ghi ở trường `form`.
- **Màu ớt giữ riêng** (ớt đỏ ≠ ớt xanh). Bỏ các cụm sơ chế ("bóc vỏ", "pha loãng", "ngâm nở"...).
- **Giữ công thức khi 100% nguyên liệu `core` ghép được** (option A). Gia vị cơ bản (`pantry`) và `optional` được bỏ qua.
- **Tạm cắt scope:** chưa loại trừ đồ người dùng đã có ("tôi có thịt gà và trứng"), chưa làm thẻ công thức, chưa dùng Gemini.

```
recipes.json ─► [1] chuẩn hóa tên ─► recipes_clean.json ──┐
catalog.json ─► [2] chuẩn hóa catalog ─► catalog_clean.json ┤
                                    [3] bge-m3 gợi ý gộp/ghép ─► CSV ─► duyệt tay ─► ALIASES + bảng ghép
                                    [4] ghép ingredient → catalog, lọc công thức đủ nguyên liệu
                                    [5] quy đổi đơn vị + mua dư (làm tròn lên gói)
                                    [6] embed công thức, truy vấn top-5   (hoãn)
```

In [1]:
import sys, json, re, collections
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))   # để import code của pipeline

RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"

def table(rows, header):
    # In bảng markdown cho dễ đọc
    from IPython.display import Markdown, display
    lines = ["| " + " | ".join(header) + " |", "|" + " :--|" * len(header)]
    lines += ["| " + " | ".join(str(c).replace("|", "\\|") for c in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))

## 1. Dữ liệu thô và vì sao phải chuẩn hóa

Hai nguồn: `recipes.json` (~2.500 món crawl từ monngonmoingay.com) và `products_clean_recipes.json` (catalog đã lọc ra hàng nấu ăn được).
Hai bên **không nói cùng một ngôn ngữ**: công thức ghi tên chung ("cà rốt", "thịt bò"), catalog ghi mặt hàng cụ thể ("Cà rốt 300g", "Thịt bò hai lát Vissan hộp 170g").

In [2]:
recipes_raw = json.loads((RAW / "recipes" / "recipes.json").read_text(encoding="utf-8"))
catalog_raw = json.loads((RAW / "catalogs" / "products_clean_recipes.json").read_text(encoding="utf-8"))

lines = [i for r in recipes_raw for i in r["ingredients"]]
distinct_raw = {i["name"].strip().lower() for i in lines}
no_amount = sum(i["amount"] is None for i in lines)
print(f"công thức: {len(recipes_raw)} | sản phẩm catalog: {len(catalog_raw)}")
print(f"dòng nguyên liệu thô: {len(lines)} | tên khác nhau: {len(distinct_raw)} | thiếu số lượng: {no_amount/len(lines):.0%}")

công thức: 2527 | sản phẩm catalog: 1129
dòng nguyên liệu thô: 29615 | tên khác nhau: 10311 | thiếu số lượng: 59%


Các kiểu nhiễu quan sát được trong dòng nguyên liệu thô: tiền tố `ĂN KÈM:`, nhiều nguyên liệu trong một dòng, `1M`/`1m` (muỗng canh/cà phê),
phân số `½`, tên thương hiệu Ajinomoto, mô tả sơ chế (băm, bóc vỏ...), cụm `A/B` và `A hoặc B`.

## 2. Bước 1: chuẩn hóa tên nguyên liệu (`scripts/build_recipes_clean.py`)

**Ý tưởng:** biến mỗi dòng thô thành một hoặc nhiều nguyên liệu dạng `{name, form, quantity, unit, role, alt}`, trong đó `name` là *khóa ghép* với catalog.
Mỗi bước là một hàm nhỏ để giải thích được từng dòng:

| Hàm | Làm gì |
| :-- | :-- |
| `strip_noise` | bỏ `®™*`, dấu ngoặc kép, gọn khoảng trắng |
| `split_prefix` | tách `ĂN KÈM:`/`RAU NÊM:` → `role="side"` |
| `split_head_tail` | tách phần tên và phần số lượng/ghi chú |
| `canonical` | bỏ thương hiệu, bỏ sơ chế, tách `form` dễ làm, bỏ mô tả, áp bảng `ALIASES` |
| `parse_quantity` | đọc số lượng: ưu tiên dữ liệu crawler, rồi `250g`, rồi `1M`, rồi `½ trái` |
| `clean_ingredient` | ghép các bước, tách dòng danh sách, tách `A/B` vào `alt` |

**Quy tắc "đổi bản chất":** dạng khó làm tại nhà là mặt hàng riêng, giữ trong tên; dạng dễ làm thì gộp về tên gốc.

In [3]:
from build_recipes_clean import clean_ingredient

samples = [
    'Hạt nêm Aji-ngon® Heo\t6g (1.5m)',
    'Nước tương "Phú Sĩ" giảm muối',
    'ĂN KÈM: Dưa kiệu, rau thơm, dưa leo cắt sợi',
    'Tương ớt, muối, tiêu',
    'Hành tây:\t1/2 củ (40g)',
    'Thơm: ½ trái',
    'Tôm khô: 20g',
    'Tỏi phi',
    'Ớt chuông đỏ     1/2 trái',
    'Ớt hiểm xanh đỏ',
    'Tôm sú bóc vỏ\t200g',
    'Bột năng pha loãng',
    'Cà hộp',
    'Húng lủi/húng bạc hà',
]
rows = []
for s in samples:
    for r in clean_ingredient({"raw_text": s, "name": s, "amount": None, "unit": None}):
        rows.append([repr(s), r["name"], r["form"] or "", r["quantity"] if r["quantity"] is not None else "",
                     r["unit"] or "", r["role"], ", ".join(r.get("alt", []))])
table(rows, ["dòng thô", "name", "form", "quantity", "unit", "role", "alt"])

| dòng thô | name | form | quantity | unit | role | alt |
| :--| :--| :--| :--| :--| :--| :--|
| 'Hạt nêm Aji-ngon® Heo\t6g (1.5m)' | hạt nêm |  | 6.0 | g | main |  |
| 'Nước tương "Phú Sĩ" giảm muối' | nước tương |  |  |  | main |  |
| 'ĂN KÈM: Dưa kiệu, rau thơm, dưa leo cắt sợi' | dưa kiệu |  |  |  | side |  |
| 'ĂN KÈM: Dưa kiệu, rau thơm, dưa leo cắt sợi' | rau thơm |  |  |  | side |  |
| 'ĂN KÈM: Dưa kiệu, rau thơm, dưa leo cắt sợi' | dưa leo |  |  |  | side |  |
| 'Tương ớt, muối, tiêu' | tương ớt |  |  |  | main |  |
| 'Tương ớt, muối, tiêu' | muối |  |  |  | main |  |
| 'Tương ớt, muối, tiêu' | tiêu |  |  |  | main |  |
| 'Hành tây:\t1/2 củ (40g)' | hành tây |  | 40.0 | g | main |  |
| 'Thơm: ½ trái' | thơm |  | 0.5 | quả | main |  |
| 'Tôm khô: 20g' | tôm khô |  | 20.0 | g | main |  |
| 'Tỏi phi' | tỏi | phi |  |  | main |  |
| 'Ớt chuông đỏ     1/2 trái' | ớt chuông đỏ |  | 0.5 | quả | main |  |
| 'Ớt hiểm xanh đỏ' | ớt hiểm xanh |  |  |  | main |  |
| 'Ớt hiểm xanh đỏ' | ớt hiểm đỏ |  |  |  | main |  |
| 'Tôm sú bóc vỏ\t200g' | tôm sú |  | 200.0 | g | main |  |
| 'Bột năng pha loãng' | bột năng |  |  |  | main |  |
| 'Cà hộp' | cà chua hộp |  |  |  | main |  |
| 'Húng lủi/húng bạc hà' | húng lủi |  |  |  | main | húng bạc hà |

Đọc bảng trên: "tôm khô" giữ nguyên (hàng riêng), "tỏi phi" → `tỏi` + `form=phi` (làm tại nhà được), "ớt hiểm xanh đỏ" tách thành hai mặt hàng,
"bóc vỏ" và "pha loãng" bị bỏ vì là bước sơ chế, "cà hộp" → "cà chua hộp".

In [4]:
clean = json.loads((PROC / "recipes_clean.json").read_text(encoding="utf-8"))
L = [i for r in clean for i in r["ingredients"]]
names = collections.Counter(i["name"] for i in L)
covered = sum(c for c in names.values() if c >= 5) / len(L)

print(f"dòng thô {len(lines)}  →  dòng sạch {len(L)}  (tăng vì dòng danh sách được tách ra)")
print(f"tên khác nhau: {len(distinct_raw)}  →  {len(names)}")
print(f"{sum(1 for c in names.values() if c >= 5)} tên xuất hiện ≥5 lần, phủ {covered:.0%} số dòng  → duyệt tay được")
print("có số lượng:", f"{sum(i['quantity'] is not None for i in L)/len(L):.0%}",
      "| role=side:", sum(i['role'] == 'side' for i in L),
      "| có alt:", sum(1 for i in L if i.get('alt')))
print("form:", dict(collections.Counter(i['form'] for i in L if i['form'])))
print("công thức không còn nguyên liệu nào:", sum(1 for r in clean if not r['ingredients']))
print("top 15:", names.most_common(15))

dòng thô 29615  →  dòng sạch 36464  (tăng vì dòng danh sách được tách ra)
tên khác nhau: 10311  →  3158
654 tên xuất hiện ≥5 lần, phủ 90% số dòng  → duyệt tay được
có số lượng: 44% | role=side: 2151 | có alt: 84
form: {'phi': 300, 'rang': 399, 'xay': 286}
công thức không còn nguyên liệu nào: 5
top 15: [('dầu ăn', 1613), ('hạt nêm', 1498), ('đường', 1457), ('tiêu', 1333), ('tỏi', 1173), ('bột ngọt', 1050), ('muối', 1049), ('hành tím', 889), ('nước tương', 822), ('nước mắm', 719), ('hành lá', 667), ('ớt sừng', 617), ('xốt mayonnaise', 610), ('cà rốt', 596), ('hành tây', 534)]


### Tự đánh giá bước 1

**Đã kiểm chứng (bằng mắt, trên mẫu):** xem ngẫu nhiên ~30 dòng raw → clean hai lần, và đo các nhóm lỗi trước/sau. Kết quả: dấu thừa trong tên 171 → 1, tên dính số lượng → 0,
mảnh "đỏ/vàng" rời 49 → 0, phân số mất số lượng 47 → 4.

**Chưa kiểm chứng:** *không có tập nhãn chuẩn*, nên chưa có số precision/recall thật. Mọi nhận xét "sạch hơn" là nhận xét trên mẫu nhỏ.

**Lỗi và rủi ro đã biết:**
- `PREP_RE` cắt tại chữ *thái*, nên "Xốt thái sả tắc" (có thể là "Thái") thành "xốt".
- Còn mô tả sót lại: "hành lá tỉa hoa", "tỏi đâp giập", "cà chua tán".
- Dòng danh sách có chung dạng ("dưa leo, củ cải và cà rốt ngâm chua ngọt") chỉ gán dạng cho một phần.
- Chỉ nhận ra thương hiệu Ajinomoto, các hãng khác còn nằm trong tên ("xốt mè aji-xốt").
- Bảng `ALIASES` mới khoảng 25 dòng. Các cặp đồng nghĩa còn lại sẽ dựa vào bước 3.
- 5 công thức không còn nguyên liệu nào cần xem lại nguồn.
- Quy tắc "dễ làm / khó làm" là phán đoán của người, chưa có ngoại lệ được duyệt.

**Cách nâng cấp:**
1. Gán nhãn ~200 dòng làm **tập chuẩn + unit test**, để sửa luật mà không sợ hỏng chỗ khác.
2. Đưa bảng luật sang file YAML để cả nhóm sửa mà không đụng code.
3. Dùng **từ vựng catalog làm tín hiệu ngược**: bỏ dần từ đuôi cho tới khi khớp một loại hàng.
4. Phần đuôi dài (6% dòng) có thể để LLM nhỏ chuẩn hóa một lần, rồi lưu cache.

## 3. Bước 2: chuẩn hóa catalog (`scripts/build_catalog_clean.py`)

**Ý tưởng:** muốn ghép được thì hai phía phải đi qua *cùng một bộ luật*. Script này tái dùng hàm `canonical` của bước 1 và thêm phần riêng cho sản phẩm:

- tách **quy cách** `pack_size`, `pack_unit`, `container` (`gói 200g`, `chai 1 lít`);
- nhận diện **bán nhiều đơn vị** (`18 chai...`, `Thùng 12 chai...`) → `multipack`;
- đoán **hãng** theo cụm chữ viết hoa liền nhau (trường `brand` gốc luôn rỗng);
- `base` = loại hàng đã chuẩn hóa, dùng để ghép với tên nguyên liệu.

In [5]:
from build_catalog_clean import clean_product

rows = []
for p in catalog_raw[:400:40]:
    c = clean_product(p)
    rows.append([c["name"][:50], c["base"], c["form"] or "", c["brand"] or "", c["pack_size"], c["pack_unit"], c["container"] or "", c["multipack"] or ""])
table(rows, ["tên gốc", "base", "form", "brand (đoán)", "pack_size", "pack_unit", "container", "multipack"])

| tên gốc | base | form | brand (đoán) | pack_size | pack_unit | container | multipack |
| :--| :--| :--| :--| :--| :--| :--| :--|
| Nước mắm Nam Ngư Phú Quốc đậm đặc 32 độ đạm chai 5 | nước mắm đậm đặc |  | Nam Ngư Phú Quốc | 500.0 | ml | chai |  |
| Dầu đậu nành Meizan chai 2 lít | dầu đậu nành |  | Meizan | 2.0 | lít | chai |  |
| Bột năng đa dụng Miwon gói 400g | bột năng đa dụng |  | Miwon | 400.0 | g | gói |  |
| Sữa tươi ít đường Dalat Milk 220ml | sữa tươi ít đường |  | Dalat Milk | 220.0 | ml |  |  |
| Kem đặc Vinamilk Tài Lộc lon 380g | kem đặc |  | Vinamilk Tài Lộc | 380.0 | g | lon |  |
| Cháo yến hải sản rong biển Yến Việt 50g | cháo yến hải sản rong biển |  | Yến Việt | 50.0 | g |  |  |
| Mì Kokomi tôm chua cay chanh ớt gói 75g | mì tôm chua cay chanh ớt |  | Kokomi | 75.0 | g | gói |  |
| Mì không cay Jin Ottogi gói 120g | mì không cay |  | Jin Ottogi | 120.0 | g | gói |  |
| Bún giò heo ăn liền Hằng Nga 75g | bún giò heo ăn liền |  | Hằng Nga | 75.0 | g |  |  |
| Chanh không hạt túi 250g (4 trái) | chanh không hạt |  |  | 250.0 | g | túi |  |

In [6]:
cat = json.loads((PROC / "catalog_clean.json").read_text(encoding="utf-8"))
print(f"sản phẩm: {len(catalog_raw)} → {len(cat)} (bỏ trùng tên + quy cách)")
print(f"có pack_size: {sum(c['pack_size'] is not None for c in cat)} | có brand đoán: {sum(c['brand'] is not None for c in cat)} | multipack: {sum(c['multipack'] is not None for c in cat)}")
print(f"loại hàng (base) khác nhau: {len({c['base'] for c in cat})}")
print("sản phẩm không có quy cách:", sum(c['pack_size'] is None for c in cat), "(chủ yếu rau củ, trái cây bán theo cân)")

sản phẩm: 1129 → 1116 (bỏ trùng tên + quy cách)
có pack_size: 990 | có brand đoán: 891 | multipack: 73
loại hàng (base) khác nhau: 895
sản phẩm không có quy cách: 126 (chủ yếu rau củ, trái cây bán theo cân)


### Tự đánh giá bước 2

**Đã kiểm chứng:** xem 22 mẫu ngẫu nhiên, sửa lỗi `base="thùng"` (34 sản phẩm) bằng cách nhận cả dạng "Thùng 12 chai...".
Lỗi thứ hai (`base` còn ngoặc, ví dụ "chanh không hạt (4 )") được phát hiện khi viết notebook này và đã sửa. Điều đó cho thấy 22 mẫu chưa đủ để tin cậy.

**Chưa kiểm chứng:** *độ đúng của `base` trên toàn bộ 1.116 sản phẩm* và độ đúng của `brand`.

**Rủi ro đã biết:**
- `brand` là đoán: "Đức La Cusina" chứa cả chữ "Đức" (nơi sản xuất), "Nam Vang" bị coi là hãng.
- Cùng lỗi chữ *thái* như bước 1, vì dùng chung `canonical`.
- 126 sản phẩm không có quy cách, mà đây lại là rau củ, đúng nhóm cần cho công thức. **Bước quy đổi (mục 6) sẽ phải xử lý, nếu không không tính được số gói.**
- Catalog hiện là dữ liệu crawl từ Bách Hóa Xanh, **chưa có ID Medusa**. Cần một bước ánh xạ khi import.

**Cách nâng cấp:** ghép `brand` với một danh sách hãng thật (lấy từ các hãng xuất hiện ≥3 lần), tách riêng xuất xứ ("Đà Lạt", "Nam Phi"), và với rau củ bán theo cân thì lưu thêm trọng lượng thật khi import vào Medusa.

## 4. Phép đo thô: độ phủ nếu chỉ khớp bằng chữ

Chưa phải ghép chính thức. Cell này chỉ trả lời: *sau chuẩn hóa, bao nhiêu dòng nguyên liệu có loại hàng tương ứng trong catalog?*
Quy tắc: tên nguyên liệu khớp `base` đúng hoặc là tiền tố của `base`, hoặc trùng tên danh mục.

In [7]:
bases = [p["base"] for p in cat]
cats = [p["category"].lower() for p in cat]

def hit(n):
    return any(x == n or x.startswith(n + " ") for x in bases) or any(n == c for c in cats)

total = sum(names.values())
matched = sum(c for n, c in names.items() if hit(n))
print(f"độ phủ theo dòng (khớp chữ thô): {matched/total:.1%}   (trước chuẩn hóa, phép khớp substring thô cho ~30%)")
print("nguyên liệu phổ biến chưa khớp:", [(n, c) for n, c in names.most_common(60) if not hit(n)][:20])

độ phủ theo dòng (khớp chữ thô): 68.6%   (trước chuẩn hóa, phép khớp substring thô cho ~30%)
nguyên liệu phổ biến chưa khớp: [('hành boa rô', 226), ('dầu điều', 212), ('tôm sú', 186), ('ngò gai', 181), ('nước cốt chanh', 171), ('bột tẩm khô chiên giòn', 168), ('rượu trắng', 148), ('đậu phộng', 136), ('nước dừa tươi', 127), ('húng lủi', 111), ('tiêu xanh', 110), ('cơm trắng', 103), ('rau om', 97), ('cần tây', 89)]


### Tự đánh giá mục 4

Độ phủ tăng từ khoảng 30% lên khoảng 69%, cho thấy việc chuẩn hóa **có tác dụng**. Nhưng cần đọc đúng:

- Đây là độ phủ **theo dòng**, không phải theo công thức. Một công thức chỉ dùng được khi **mọi** nguyên liệu `core` của nó khớp, nên tỉ lệ công thức đủ nguyên liệu sẽ thấp hơn nhiều. **Con số này chưa được đo** (bước 4).
- Khớp tiền tố có thể khớp *sai*: "muối" khớp "muối tôm", "tiêu" khớp "tiêu xay". Phép đo này chưa có kiểm tra đúng/sai.
- Các nguyên liệu chưa khớp (hành boa rô, dầu điều, tôm sú, ngò gai...) phần lớn là hàng **catalog chưa bán**, không phải do tên sai. Đây là giới hạn của dữ liệu, không sửa bằng luật được.

## 5. Bước 3: bge-m3 gợi ý gộp tên và ghép catalog (`scripts/build_alias_candidates.py`)

**Ý tưởng:** embed (vector 1024 chiều) toàn bộ tên nguyên liệu và `base` của catalog bằng `BAAI/bge-m3`, tính cosine, rồi **đề xuất**:

1. `alias_candidates.csv`: tên hiếm → tên phổ biến giống nó (≥ 0,85), gắn nhãn `variant` hoặc `synonym?`;
2. `catalog_candidates.csv`: mỗi nguyên liệu (≥ 5 lần) → top-3 loại hàng catalog.

**Script không tự gộp.** bge-m3 coi "ớt đỏ" và "ớt xanh" rất giống nhau, mà theo quyết định thì hai thứ này khác nhau. Người duyệt đánh dấu cột `accept`,
dòng đúng được chép vào `ALIASES`.

In [8]:
import csv
ap, cp = PROC / "alias_candidates.csv", PROC / "catalog_candidates.csv"
if ap.exists() and cp.exists():
    alias = list(csv.DictReader(ap.open(encoding="utf-8-sig")))
    cands = list(csv.DictReader(cp.open(encoding="utf-8-sig")))
    syn = [r for r in alias if r["relation"] == "synonym?"]
    print(f"alias_candidates: {len(alias)} dòng ({len(syn)} nhãn synonym?, {len(alias)-len(syn)} variant)")
    table([[r["name"], r["count"], r["merge_into"], r["target_count"], r["similarity"]] for r in syn[:15]],
          ["name", "count", "merge_into", "target_count", "similarity"])
    top1 = [r for r in cands if r["rank"] == "1"]
    lex = sum(1 for r in top1 if r["lexical"])
    print(f"catalog_candidates: {len(top1)} nguyên liệu | top-1 trùng khớp chữ (exact/prefix): {lex}")
    table([[r["name"], r["count"], r["catalog_base"], r["category"], r["similarity"], r["lexical"]] for r in top1[:15]],
          ["name", "count", "top-1 catalog base", "category", "similarity", "lexical"])
else:
    print("CHƯA CÓ kết quả: alias_candidates.csv / catalog_candidates.csv chưa được tạo.")
    print("Chạy:  .venv\\Scripts\\python scripts\\build_alias_candidates.py   rồi chạy lại cell này.")

CHƯA CÓ kết quả: alias_candidates.csv / catalog_candidates.csv chưa được tạo.
Chạy:  .venv\Scripts\python scripts\build_alias_candidates.py   rồi chạy lại cell này.


### Tự đánh giá bước 3

*(Phần này sẽ cập nhật sau khi có kết quả chạy.)* Những điều cần kiểm tra khi có file:

- Thời gian embed và kích thước vector trên CPU (issue #8 yêu cầu ghi lại).
- Với các nguyên liệu mà phép khớp chữ đã đúng, top-1 của bge-m3 có trùng không? Đây là phép kiểm tra rẻ nhất cho độ tin cậy.
- Ngưỡng 0,85 là **đoán, chưa tune**. Cần xem phân bố similarity của các cặp đã duyệt đúng/sai để chọn ngưỡng.
- Rủi ro biết trước: embedding của chuỗi ngắn ("ớt đỏ" vs "ớt xanh", "tôm" vs "tôm khô") khó phân biệt các khác biệt nhỏ nhưng quan trọng.

**Cách nâng cấp:** lọc theo `category` trước rồi mới so similarity (docs/05 mục D5); dùng bge-m3 *sparse/lexical weights* hoặc reranker `bge-reranker-v2-m3`;
so với model nhỏ hơn (multilingual-e5, vietnamese-sbert) trên cùng tập nhãn; lưu vector vào pgvector thay vì `.npy` khi dựng `ai_db`.

## 6. Bước 5 (chưa làm): quy đổi đơn vị và quy tắc **mua dư**

Catalog là nguồn sự thật, không đổi đơn vị. Vì vậy công thức phải đổi về đơn vị catalog rồi làm tròn **lên** theo cỡ gói:

```
nhu_cau  = so_luong_trong_cong_thuc × (so_nguoi_yeu_cau / so_nguoi_chuan)     # đơn vị công thức
nhu_cau_g = nhu_cau × gram_moi_don_vi          # bảng quy đổi: 1 trái cà chua ≈ 40g, 1 muỗng canh ≈ 15g...
so_goi    = ceil(nhu_cau_g / pack_size_g)      # luôn ≥ 1 và là số nguyên
```

Cell dưới là **minh họa thiết kế**, *không nằm trong pipeline* và dùng số giả định (40g/trái, gói 500g) để xem cách tính.

In [9]:
import math

def packs_needed(qty, gram_per_unit, pack_g, servings_req=4, servings_std=4):
    # Minh họa: nhu cầu quy ra gam, làm tròn lên số gói (mua dư).
    need_g = qty * gram_per_unit * servings_req / servings_std
    return max(1, math.ceil(need_g / pack_g)), need_g

for label, qty, gpu, pack in [("cà chua: 3 trái, gói 500g", 3, 40, 500),
                              ("cà chua: 3 trái, gói 500g, nấu cho 8 người", 3, 40, 500),
                              ("thịt ba chỉ: 300g, khay 500g", 300, 1, 500),
                              ("nước mắm: 2 muỗng canh (15g), chai 500ml", 2, 15, 500)]:
    n, need = packs_needed(qty, gpu, pack, servings_req=8 if "8 người" in label else 4)
    print(f"{label:48s} cần {need:.0f}g → mua {n} gói")

cà chua: 3 trái, gói 500g                        cần 120g → mua 1 gói
cà chua: 3 trái, gói 500g, nấu cho 8 người       cần 240g → mua 1 gói
thịt ba chỉ: 300g, khay 500g                     cần 300g → mua 1 gói
nước mắm: 2 muỗng canh (15g), chai 500ml         cần 30g → mua 1 gói


### Việc phải làm ở bước này, và rủi ro

- **Bảng quy đổi** (`unit_conversions.json`, có version): trái/quả/củ/cây/nhánh/muỗng → gam, **theo từng nguyên liệu** (1 trái cà chua ≠ 1 trái thơm). Đây là phần cần người viết, không tự động được.
- **Dòng không có số lượng:** ~37% dòng không phải gia vị cơ bản (tỏi, hành tím, hành lá, ngò rí...). Cần quy tắc: gán số lượng mặc định theo nhóm, hay coi là "nêm nếm" và không đưa vào giỏ.
- **Sản phẩm không có quy cách** (rau củ bán theo cân): cần cỡ gói mặc định.
- **Mua dư** là lựa chọn có chủ đích: đảm bảo *đủ*, đổi lại có thể thừa. Nên hiển thị lượng cần so với lượng mua để người dùng hiểu.
- Số gói cần được **tính lại khi khách đổi khẩu phần**, không nên lưu cứng.

## 7. Các bước tiếp theo

**Bước 4: đo số công thức đủ nguyên liệu (option A).** Sau khi duyệt bảng gộp và bảng ghép, với mỗi công thức kiểm tra: mọi nguyên liệu `core` (không `pantry`, không `optional`, không `side`) đều ghép được ít nhất một sản phẩm.
Chưa có con số. Đây là con số quan trọng nhất để biết MVP có đủ công thức để demo không. Nếu quá ít, thử option C (cho phép thiếu vài nguyên liệu và báo rõ).
Cần thêm cờ `pantry` và `optional`, hiện mới có `role` (main/side).

**Bước 6 (hoãn): embed công thức và truy vấn.** Khi làm: embed *thẻ công thức* ngắn (tên món, mô tả, vài nguyên liệu chính) thay vì cả công thức; tách số người khỏi câu hỏi;
cần ngưỡng similarity để báo "không tìm thấy"; thử truy vấn không dấu. Đánh giá bằng Recall@K trên bộ 50–100 truy vấn tự gán nhãn, so với tìm bằng chữ (ILIKE), đúng như giả thuyết H5.

**Ngoài scope hiện tại:** loại trừ đồ người dùng đã có, Gemini, đồng bộ với Medusa.

## 8. Tổng kết đánh giá

| Mục | Mức tin cậy | Lý do |
| :-- | :-: | :-- |
| Chuẩn hóa tên công thức | Trung bình | Số liệu trước/sau rõ, nhưng chỉ kiểm bằng mắt trên mẫu, chưa có tập nhãn |
| Chuẩn hóa catalog | Trung bình-thấp | Mới xem ~22 mẫu, `brand` là đoán |
| Độ phủ 69% | Thấp | Khớp chữ thô, chưa kiểm đúng/sai, và chưa phải độ phủ theo công thức |
| bge-m3 gợi ý | Chưa đánh giá | Chưa có kết quả, chưa duyệt tay |
| Quy tắc mua dư | Thiết kế | Chưa cài đặt, chưa có bảng quy đổi |

**Điểm yếu lớn nhất của pipeline hiện tại:** *không có tập nhãn chuẩn nào*. Mọi bước đều đang "trông có vẻ đúng". Nên gán nhãn một tập nhỏ (200 dòng nguyên liệu và 100 cặp ingredient → sản phẩm) trước khi tune thêm bất cứ ngưỡng nào.

**Phụ thuộc vào dữ liệu:** nhiều nguyên liệu chưa khớp là do catalog thiếu hàng (đặc biệt đồ tươi). Nếu số công thức đủ nguyên liệu thấp, nguyên nhân có thể nằm ở catalog chứ không phải pipeline.

**Lưu ý đạo văn và demo:** thầy sẽ hỏi từng dòng code của tính năng demo. Giữ luật đơn giản, đặt tên rõ, và phần văn xuôi nộp cho thầy cần nhóm tự viết lại.